# Clean & Merge SFWMD Flow Data

**Part 1 — Clean:** Reads raw SFWMD FLOW CSVs, resamples to hourly mean, and writes a UTC-indexed clean CSV.

**Part 2 — Merge:** Joins the clean flow column into the main merged dataset.

| Step | Input | Output |
|------|-------|--------|
| Clean | `data/FLOW/FLOW_S28_S/*.csv` | `data/FLOW/clean/FLOW_S28_S_clean.csv` |
| Merge | `data/Merged/Miami_GWL_WL_RAIN_GATE_2017_2024.csv` + clean flow | `data/Merged/Miami_GWL_WL_RAIN_GATE_FLOW_2017_2024.csv` |

Import standard libraries for file globbing, path operations, and tabular data manipulation.

In [1]:
import glob
import os
import pandas as pd

## Part 1: Clean Raw Flow Data

Define the glob pattern for raw SFWMD CSVs, the output path for the clean file, and the number of metadata rows to skip at the top of each raw file.

In [2]:
RAW_PATTERN = "../data/FLOW/FLOW_S28_S/*.csv"
FLOW_CLEAN  = "../data/FLOW/clean/FLOW_S28_S_clean.csv"
HEADER_ROWS = 34  # SFWMD warning/metadata rows before the CSV header

Define the cleaning function. It reads all matched CSVs, coerces the `VALUE` column to numeric (silencing non-numeric entries), concatenates them into one time series, and resamples to an hourly mean with a UTC-localized index.

In [3]:
def clean_sfwmd_to_hourly(file_pattern, column_name, header_rows=34):
    files = sorted(glob.glob(file_pattern))
    if not files:
        raise FileNotFoundError(f"No files matched pattern: {file_pattern}")

    print(f"Found {len(files)} file(s):")
    for f in files:
        print(f"  {f}")

    dfs = []
    for f in files:
        df = pd.read_csv(f, skiprows=header_rows, usecols=["TIMESTAMP", "VALUE"], low_memory=False)
        df["VALUE"] = pd.to_numeric(df["VALUE"], errors="coerce")
        df["TIMESTAMP"] = pd.to_datetime(df["TIMESTAMP"])
        dfs.append(df)

    combined = pd.concat(dfs, ignore_index=True).set_index("TIMESTAMP").sort_index()
    print(f"\nTotal rows before resampling: {len(combined)}")

    hourly = combined["VALUE"].resample("h").mean()
    hourly.index = hourly.index.tz_localize("UTC")
    hourly.name = column_name

    return hourly.to_frame()

Run the cleaning function and print a summary of the resulting hourly DataFrame — row count, date range, and missing value count — then preview the first five rows.

In [4]:
df_clean = clean_sfwmd_to_hourly(RAW_PATTERN, column_name="flow", header_rows=HEADER_ROWS)

print(f"Hourly rows: {len(df_clean)}")
print(f"Date range: {df_clean.index[0]} \u2192 {df_clean.index[-1]}")
print(f"NaN count: {df_clean['flow'].isna().sum()}")
df_clean.head(5)

Found 7 file(s):
  ../data/FLOW/FLOW_S28_S/FLOW_S28_S_20171001_20181231.csv
  ../data/FLOW/FLOW_S28_S/FLOW_S28_S_20190101_20200229.csv
  ../data/FLOW/FLOW_S28_S/FLOW_S28_S_20200301_20201231.csv
  ../data/FLOW/FLOW_S28_S/FLOW_S28_S_20210101_20211231.csv
  ../data/FLOW/FLOW_S28_S/FLOW_S28_S_20220101_20221231.csv
  ../data/FLOW/FLOW_S28_S/FLOW_S28_S_20230101_20231231.csv
  ../data/FLOW/FLOW_S28_S/FLOW_S28_S_20240101_20241231.csv

Total rows before resampling: 1626749
Hourly rows: 63576
Date range: 2017-10-01 00:00:00+00:00 → 2024-12-31 23:00:00+00:00
NaN count: 3


,flow
TIMESTAMP,
2017-10-01 00:00:00+00:00,568.957105
2017-10-01 01:00:00+00:00,295.402519
2017-10-01 02:00:00+00:00,13.024500
2017-10-01 03:00:00+00:00,0.000000
2017-10-01 04:00:00+00:00,0.000000


Create the output directory if it does not exist, then write the clean hourly DataFrame to CSV.

In [5]:
os.makedirs(os.path.dirname(FLOW_CLEAN), exist_ok=True)
df_clean.to_csv(FLOW_CLEAN)
print(f"Saved \u2192 {FLOW_CLEAN}")

Saved → ../data/FLOW/clean/FLOW_S28_S_clean.csv


## Part 2: Merge Flow into Main Dataset

Define paths for the existing merged dataset (input) and the new merged dataset with flow appended (output).

In [6]:
MERGED_IN  = "../data/Merged/Miami_GWL_WL_RAIN_GATE_2017_2024.csv"
MERGED_OUT = "../data/Merged/Miami_GWL_WL_RAIN_GATE_FLOW_2017_2024.csv"

Load both datasets, ensure the merged index is UTC-localized to align with the flow index, then left-join the flow column. Print column list, shape, and flow NaN count to confirm the join worked as expected, and preview gate and flow columns together.

In [10]:
merged = pd.read_csv(MERGED_IN, index_col=0, parse_dates=True)
flow   = pd.read_csv(FLOW_CLEAN, index_col=0, parse_dates=True)

if merged.index.tz is None:
    merged.index = merged.index.tz_localize("UTC")

merged = merged.join(flow, how="left")

print(f"Columns:        {merged.columns.tolist()}")
print(f"Shape:          {merged.shape}")
print(f"Flow NaN count: {merged['flow'].isna().sum()} / {len(merged)}")
merged[["gwl", "wl", "flow"]].head(5)

Columns:        ['gwl', 'wl', 'rain', 'stgH', 'stgT', 'gate1', 'gate2', 'flow']
Shape:          (70134, 8)
Flow NaN count: 6561 / 70134


,gwl,wl,flow
2017-01-01 00:00:00+00:00,NaN,-0.239,NaN
2017-01-01 01:00:00+00:00,NaN,-0.123,NaN
2017-01-01 02:00:00+00:00,NaN,-0.001,NaN
2017-01-01 03:00:00+00:00,NaN,0.080,NaN
2017-01-01 04:00:00+00:00,NaN,0.102,NaN


Write the merged DataFrame — now including the flow column — to a new CSV.

In [8]:
merged.to_csv(MERGED_OUT)
print(f"Saved \u2192 {MERGED_OUT}")

Saved → ../data/Merged/Miami_GWL_WL_RAIN_GATE_FLOW_2017_2024.csv
